# Лабораторна робота № 3 — моделювання систем з використанням мереж Петрі

Варіант 3. Вовкотруб Олександр Віталійович, група ФБ-61мн.
КПІ ім. Ігоря Сікорського, ННФТІ, кафедра інформаційної безпеки.

Аналіз заданої мережі (граф досяжних розміток, P-інваріанти) та синтез
моделі задачі про обідаючих філософів.
Зведення обчислених величин наприкінці вивантажується у `data/results-summary.csv`.

## 0. Підготовка середовища

Клітинка нижче створює теки `assets/` і `data/` поруч із ноутбуком і, якщо
якоїсь бібліотеки бракує, доставляє її. Завдяки цьому ноутбук виконується
як локально, так і в Colab без жодних ручних кроків.
У Colab додатково встановлюється системний Graphviz — саме ним
рендеряться зображення мереж Петрі.

In [1]:
import importlib.util
import shutil
import subprocess
import sys
from pathlib import Path


def _in_colab() -> bool:
    if "google.colab" in sys.modules:
        return True
    try:
        return importlib.util.find_spec("google.colab") is not None
    except (ImportError, ValueError):
        # find_spec імпортує батьківський пакет; без нього це не Colab
        return False


IN_COLAB = _in_colab()

REQUIRED = ['numpy', 'sympy', 'matplotlib', 'graphviz']
missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Graphviz — це ще й системний бінарник `dot`, не лише пакет Python.
if IN_COLAB and shutil.which("dot") is None:
    subprocess.run(["apt-get", "-qq", "install", "-y", "graphviz"], check=True)

ASSETS = Path("assets")
DATA = Path("data")
ASSETS.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

### Симулятор мереж Петрі

Рушій, на якому тримаються обидві частини роботи: спрацювання переходів із
урахуванням ваг дуг, обхід графа досяжних розміток, P-інваріанти через
нуль-простір матриці інцидентності та рендеринг у Graphviz.

In [2]:
"""Мінімальний симулятор мереж Петрі.

Спрацювання переходів із урахуванням ваг дуг, граф досяжних розміток,
P-інваріанти через нуль-простір матриці інцидентності та рендеринг у Graphviz.
Модуль нічого не знає про предметну область — моделі задач лежать окремо.
"""
from __future__ import annotations

from collections import deque
from pathlib import Path
from typing import Iterable, Mapping, Sequence

Marking = tuple[int, ...]
Arcs = Mapping[str, int]


class PetriNet:
    def __init__(self, places: Sequence[str],
                 transitions: Mapping[str, tuple[Arcs, Arcs]]) -> None:
        self.places = list(places)
        self.transitions = {t: (dict(pre), dict(post))
                            for t, (pre, post) in transitions.items()}
        self.index = {p: i for i, p in enumerate(self.places)}
        for name, (pre, post) in self.transitions.items():
            unknown = (set(pre) | set(post)) - set(self.index)
            if unknown:
                raise ValueError(f"перехід {name}: невідомі позиції {sorted(unknown)}")

    def enabled(self, m: Marking) -> list[str]:
        """Переходи, дозволені у розмітці m."""
        return [t for t, (pre, _) in self.transitions.items()
                if all(m[self.index[p]] >= w for p, w in pre.items())]

    def fire(self, m: Marking, t: str) -> Marking:
        """Розмітка після спрацювання переходу t."""
        pre, post = self.transitions[t]
        if any(m[self.index[p]] < w for p, w in pre.items()):
            raise ValueError(f"перехід {t} не дозволений у розмітці {m}")
        out = list(m)
        for p, w in pre.items():
            out[self.index[p]] -= w
        for p, w in post.items():
            out[self.index[p]] += w
        return tuple(out)

    def reachability_graph(self, m0: Marking, max_markings: int = 10_000):
        """Обхід у ширину: (список розміток у порядку відкриття, список дуг)."""
        nodes: list[Marking] = [m0]
        edges: list[tuple[Marking, str, Marking]] = []
        seen, queue = {m0}, deque([m0])
        while queue:
            m = queue.popleft()
            for t in self.enabled(m):
                nxt = self.fire(m, t)
                edges.append((m, t, nxt))
                if nxt not in seen:
                    if len(seen) >= max_markings:
                        raise RuntimeError(
                            f"понад {max_markings} розміток — мережа необмежена?")
                    seen.add(nxt)
                    nodes.append(nxt)
                    queue.append(nxt)
        return nodes, edges

    def deadlocks(self, nodes: Iterable[Marking]) -> list[Marking]:
        """Розмітки, у яких не дозволений жоден перехід."""
        return [m for m in nodes if not self.enabled(m)]

    def bounds(self, nodes: Iterable[Marking]) -> dict[str, int]:
        """Максимум фішок у кожній позиції на множині розміток."""
        nodes = list(nodes)
        return {p: max(m[i] for m in nodes) for p, i in self.index.items()}

    def incidence(self):
        """Матриця інцидентності C = post - pre розміру |P| x |T| та імена переходів."""
        import sympy as sp
        names = list(self.transitions)
        C = sp.zeros(len(self.places), len(names))
        for j, t in enumerate(names):
            pre, post = self.transitions[t]
            for p, w in pre.items():
                C[self.index[p], j] -= w
            for p, w in post.items():
                C[self.index[p], j] += w
        return C, names

    def p_invariants(self) -> list[list[int]]:
        """Базис P-інваріантів: цілочислові x, для яких x^T · C = 0."""
        import sympy as sp
        C, _ = self.incidence()
        out = []
        for vector in C.T.nullspace():
            denominators = [sp.Rational(value).q for value in vector]
            scale = sp.ilcm(*denominators) if len(denominators) > 1 else denominators[0]
            out.append([int(value * scale) for value in vector])
        return out

    def to_dot(self, m: Marking | None = None, name: str = "petri") -> str:
        """DOT-опис самої мережі; якщо задано m — з фішками в позиціях."""
        lines = [f"digraph {name} {{", "  rankdir=LR;",
                 '  node [fontname="Liberation Sans", fontsize=11];',
                 '  edge [fontname="Liberation Sans", fontsize=10];']
        for p in self.places:
            count = m[self.index[p]] if m is not None else 0
            if count == 0:
                label = p
            elif count <= 4:
                label = f"{p}\\n{'●' * count}"
            else:
                label = f"{p}\\n{count}"
            lines.append(f'  "{p}" [shape=circle, fixedsize=true, width=0.62, label="{label}"];')
        for t in self.transitions:
            lines.append(f'  "{t}" [shape=box, style=filled, fillcolor=black, '
                         f'label="", xlabel="{t}", width=0.09, height=0.44];')
        for t, (pre, post) in self.transitions.items():
            for p, w in pre.items():
                lines.append(f'  "{p}" -> "{t}"' + (f' [label="{w}"];' if w > 1 else ";"))
            for p, w in post.items():
                lines.append(f'  "{t}" -> "{p}"' + (f' [label="{w}"];' if w > 1 else ";"))
        lines.append("}")
        return "\n".join(lines)

    @staticmethod
    def graph_to_dot(nodes, edges, name: str = "reach", rankdir: str = "LR") -> str:
        """DOT-опис графа досяжних розміток."""
        def label(m: Marking) -> str:
            return "(" + ", ".join(map(str, m)) + ")"

        lines = [f"digraph {name} {{", f"  rankdir={rankdir};",
                 '  node [shape=ellipse, fontname="Liberation Mono", fontsize=10];',
                 '  edge [fontname="Liberation Sans", fontsize=9];']
        for i, m in enumerate(nodes):
            extra = ', style=filled, fillcolor="#dce6f7"' if i == 0 else ""
            lines.append(f'  "{label(m)}" [label="{label(m)}"{extra}];')
        for src, t, dst in edges:
            lines.append(f'  "{label(src)}" -> "{label(dst)}" [label="{t}"];')
        lines.append("}")
        return "\n".join(lines)


def render(dot_source: str, out_path: Path | str) -> Path:
    """Відрендерити DOT у PNG за вказаним шляхом (розширення додається саме)."""
    import graphviz
    out_path = Path(out_path)
    graphviz.Source(dot_source, format=out_path.suffix.lstrip(".") or "png") \
            .render(out_path.with_suffix(""), cleanup=True)
    return out_path

In [3]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Liberation Serif", "DejaVu Serif"],
    "font.size": 11,
    "figure.dpi": 150,
    "savefig.bbox": "tight",
    "axes.grid": True,
    "grid.alpha": 0.3,
})

RESULTS: list[dict] = []


def record(section: str, name: str, value, note: str = "") -> None:
    """Запам'ятати число для `data/results-summary.csv` і показати його."""
    RESULTS.append({"розділ": section, "показник": name, "значення": value, "примітка": note})
    print(f"{section} | {name} = {value}  {note}")

## 1. Аналіз мережі Петрі — варіант 3

Мережу задано рисунком на с. 6 умови (вирізка оригіналу —
`assets/00-net-var3-source.png`). Зчитані позиції, переходи та ваги дуг:

| Перехід | Вхід | Вихід |
|---|---|---|
| `t1` | `p4`(1), `p2`(2) | `p1`(1) |
| `t2` | `p1`(2) | `p4`(1), `p3`(1) |
| `t3` | `p3`(1) | `p1`(1), `p2`(2) |

Початкова розмітка $\mu_0 = (2, 0, 1, 0)$ у порядку $(p_1, p_2, p_3, p_4)$:
`p1` — центральна позиція (2 фішки), `p2` — нижня ліва (0), `p3` — права
(1 фішка), `p4` — верхня права (0).

In [4]:
PLACES_V3 = ["p1", "p2", "p3", "p4"]
TRANS_V3 = {
    "t1": ({"p4": 1, "p2": 2}, {"p1": 1}),
    "t2": ({"p1": 2}, {"p4": 1, "p3": 1}),
    "t3": ({"p3": 1}, {"p1": 1, "p2": 2}),
}
M0_V3 = (2, 0, 1, 0)

net = PetriNet(PLACES_V3, TRANS_V3)
render(net.to_dot(M0_V3, name="var3"), ASSETS / "01-net-var3.png")

PosixPath('assets/01-net-var3.png')

### Граф досяжних розміток

In [5]:
nodes, edges = net.reachability_graph(M0_V3)
deadlocks = net.deadlocks(nodes)
bounds = net.bounds(nodes)

record("Аналіз МП", "досяжних розміток", len(nodes))
record("Аналіз МП", "дуг у графі", len(edges))
record("Аналіз МП", "тупикових розміток", len(deadlocks))
record("Аналіз МП", "межі позицій", bounds)
record("Аналіз МП", "мережа k-обмежена, k =", max(bounds.values()),
       "не безпечна, бо k > 1")

assert len(nodes) == 7
assert len(edges) == 11
assert deadlocks == []
assert bounds == {"p1": 3, "p2": 6, "p3": 2, "p4": 2}

print("Досяжні розмітки (p1, p2, p3, p4):")
for m in nodes:
    print("  ", m, "  дозволені переходи:", ", ".join(net.enabled(m)))

render(PetriNet.graph_to_dot(nodes, edges, name="reach_var3"),
             ASSETS / "02-reachability-var3.png")

Аналіз МП | досяжних розміток = 7  
Аналіз МП | дуг у графі = 11  
Аналіз МП | тупикових розміток = 0  
Аналіз МП | межі позицій = {'p1': 3, 'p2': 6, 'p3': 2, 'p4': 2}  
Аналіз МП | мережа k-обмежена, k = = 6  не безпечна, бо k > 1
Досяжні розмітки (p1, p2, p3, p4):
   (2, 0, 1, 0)   дозволені переходи: t2, t3
   (0, 0, 2, 1)   дозволені переходи: t3
   (3, 2, 0, 0)   дозволені переходи: t2
   (1, 2, 1, 1)   дозволені переходи: t1, t3
   (2, 4, 0, 1)   дозволені переходи: t1, t2
   (0, 4, 1, 2)   дозволені переходи: t1, t3
   (1, 6, 0, 2)   дозволені переходи: t1


PosixPath('assets/02-reachability-var3.png')

Граф досяжності скінченний і повністю збудований обходом у ширину (жодна
розмітка не відкинута через ліміт), тому відсутність тупиків у ньому —
це не спостереження на вибірці, а вичерпний доказ: серед усіх 7 досяжних
розміток немає жодної, де заблоковані всі три переходи.

### P-інваріанти

Інваріант — вектор $x$, для якого $x^{T}C = 0$, де $C$ — матриця інцидентності.
Тоді $x^{T}\mu$ **однаковий у будь-якій досяжній розмітці**. Інваріанти дають
строге доведення обмеженості, яке не залежить від перебору: якщо зважена
сума фішок стала, а всі ваги в ній додатні, то кожна доданкова позиція
обмежена зверху цією сталою — обмеженість випливає зі структури мережі, а
не лише з того, що саме ці 7 розміток виявились скінченними.

In [6]:
C, trans_names = net.incidence()
print("Матриця інцидентності C (рядки — позиції, стовпці —", trans_names, "):")
print(C)

invariants = net.p_invariants()
for inv in invariants:
    expr = " + ".join(f"{w}·{p}" for w, p in zip(inv, PLACES_V3) if w)
    values = {sum(w * m[i] for i, w in enumerate(inv)) for m in nodes}
    assert len(values) == 1
    const = values.pop()
    record("Аналіз МП", f"P-інваріант: {expr}", const, "сталий у всіх розмітках")
    print(f"  {expr} = {const}")

assert len(invariants) == 2

Матриця інцидентності C (рядки — позиції, стовпці — ['t1', 't2', 't3'] ):
Matrix([[1, -2, 1], [-2, 0, 2], [0, 1, -1], [-1, 1, 0]])
Аналіз МП | P-інваріант: 2·p1 + 1·p2 + 4·p3 = 8  сталий у всіх розмітках
  2·p1 + 1·p2 + 4·p3 = 8
Аналіз МП | P-інваріант: 2·p1 + -1·p2 + 4·p4 = 4  сталий у всіх розмітках
  2·p1 + -1·p2 + 4·p4 = 4


Базис, який фактично повернув `nullspace()`, — це `2p1 + p2 + 4p3 = 8` і
`2p1 - p2 + 4p4 = 4` (обидва інваріанти надруковані вище саме в цій формі).
Це інша, але еквівалентна форма тієї, що згадана в умові: якщо додати обидва
рівняння й поділити на 4, отримаємо `p1 + p3 + p4 = 3` — комбінація тих
самих двох базисних векторів, лише зручніша для читання. З неї одразу видно,
що жодна з позицій `p1`, `p3`, `p4` не може перевищити 3 фішки (кожна входить
з додатною вагою в суму, що дорівнює 3). З першого інваріанта
`2p1 + p2 + 4p3 = 8` при `p1, p3 ≥ 0` так само одразу випливає `p2 ≤ 8` —
це структурна межа, яку дають самі інваріанти незалежно від перебору.
Фактичний максимум `p2 ≤ 6`, знайдений перебором графа досяжності, тісніший:
не кожна невід'ємна ціла розмітка, що задовольняє обом рівнянням інваріантів
(наприклад, `(0, 8, 0, 3)`), насправді досяжна з `μ₀` — інваріанти
гарантують обмеженість, але не відтворюють точну межу без побудови графа.

## 2. Синтез — задача 3 про обідаючих філософів

**Мій варіант:** «Перетворення задачі 2 про обідаючих філософів для випадку,
коли за стіл запрошують одного з філософів, який не перешкоджає загальному
поглинанню їжі обідаючими.»

Будуємо трьома кроками, щоб було видно саме *перетворення* умови, а не лише
кінцевий результат:

1. **задача 1** — філософ бере обидві виделки одним переходом (атомарно).
   Тупику немає — атомарність не залишає проміжного стану, в якому можна
   застрягнути;
2. **задача 2** — виделки беруться по черзі, через проміжну позицію
   `left_i`. З'являється класичний тупик: усі філософи одночасно тримають
   ліву виделку й чекають на праву;
3. **задача 3 (мій варіант)** — додаємо позицію `invite`, яка обмежує,
   скільки філософів одночасно можуть узяти ліву виделку. Тупик зникає.

### Скільки фішок класти в позицію `invite`?

Умову можна прочитати двояко:

* буквально «запрошують **одного**» → 1 фішка в `invite`. Але тоді за
  столом завжди щонайбільше один їдець — а це прямо суперечить продовженню
  тієї самої фрази: «**не перешкоджає загальному поглинанню їжі
  обідаючими**». Один їдець — це не «загальне поглинання їжі», а черга;
* отже фішок має бути стільки, щоб паралельне харчування збереглося, а
  тупик усе одно зник: $n - 1$. Це відомий семафорний розв'язок задачі про
  обідаючих філософів — якщо за столом одночасно щонайбільше $n - 1$
  філософів, завжди лишається хоча б один філософ, чий сусід ще не сидить
  за столом, тож ланцюжок очікування «кожен тримає ліву й чекає на праву»
  ніколи не замикається в коло.

Додатковий аргумент — формулювання **задачі 4** з того самого переліку
завдань варіанта 3: «перетворення задачі 3 для випадку, коли запрошення до
столу **необов'язкові для розгляду філософом**». Це показує, що в задачі 3 (моєму варіанті)
запрошення — не одноразова подія, а обов'язковий механізм доступу до
столу, який діє постійно; задача 4 лише знімає цю обов'язковість. Отже
`invite` повинна діяти як семафор, що витрачається на вхід і повертається
на вихід, а не як фішка, яку видали один раз і забули.

In [7]:
"""Моделі мереж Петрі для задачі про обідаючих філософів (ЛР № 3, синтез).

Три моделі, що показують саме *перетворення* умови:

* `build_task1` — обидві виделки беруться одним переходом (задача 1);
* `build_task2` — виделки беруться по черзі (задача 2) — з'являється тупик;
* `build_task3` — доданий семафор запрошень (задача 3, мій варіант) — тупик зникає.

Філософи сидять по колу: філософ i використовує виделки i та (i+1) mod n.
"""
from __future__ import annotations

try:
    from petri import Marking, PetriNet
except ModuleNotFoundError:
    # У ноутбуці код petri.py вбудовано вище окремою клітинкою, тож модуля
    # як такого немає — потрібні імена вже лежать у глобальному просторі.
    pass


def build_task1(n: int) -> tuple[PetriNet, Marking]:
    """Задача 1: обидві виделки беруться одночасно, одним переходом."""
    places = ([f"think{i}" for i in range(n)]
              + [f"eat{i}" for i in range(n)]
              + [f"fork{i}" for i in range(n)])
    transitions = {}
    for i in range(n):
        right = (i + 1) % n
        transitions[f"take{i}"] = (
            {f"think{i}": 1, f"fork{i}": 1, f"fork{right}": 1}, {f"eat{i}": 1})
        transitions[f"put{i}"] = (
            {f"eat{i}": 1}, {f"think{i}": 1, f"fork{i}": 1, f"fork{right}": 1})
    marking = tuple([1] * n + [0] * n + [1] * n)
    return PetriNet(places, transitions), marking


def build_task2(n: int) -> tuple[PetriNet, Marking]:
    """Задача 2: ліва й права виделки беруться по черзі.

    Проміжний стан `left_i` (є ліва виделка, немає правої) і робить можливим
    класичний тупик: усі n філософів тримають по лівій виделці.
    """
    places = ([f"think{i}" for i in range(n)]
              + [f"left{i}" for i in range(n)]
              + [f"eat{i}" for i in range(n)]
              + [f"fork{i}" for i in range(n)])
    transitions = {}
    for i in range(n):
        right = (i + 1) % n
        transitions[f"takeL{i}"] = ({f"think{i}": 1, f"fork{i}": 1}, {f"left{i}": 1})
        transitions[f"takeR{i}"] = ({f"left{i}": 1, f"fork{right}": 1}, {f"eat{i}": 1})
        transitions[f"put{i}"] = (
            {f"eat{i}": 1}, {f"think{i}": 1, f"fork{i}": 1, f"fork{right}": 1})
    marking = tuple([1] * n + [0] * n + [0] * n + [1] * n)
    return PetriNet(places, transitions), marking


def build_task3(n: int, invites: int | None = None) -> tuple[PetriNet, Marking]:
    """Задача 3: доступ до столу за запрошенням.

    Позиція `invite` — семафор: узяти ліву виделку можна лише витративши
    запрошення, а повертається воно наприкінці трапези. За замовчуванням
    запрошень `n - 1` — рівно стільки, щоб паралельне харчування збереглося
    ("не перешкоджає загальному поглинанню їжі"), а тупик став недосяжним.
    """
    invites = n - 1 if invites is None else invites
    base, base_marking = build_task2(n)
    places = base.places + ["invite"]
    transitions = {}
    for name, (pre, post) in base.transitions.items():
        pre, post = dict(pre), dict(post)
        if name.startswith("takeL"):
            pre["invite"] = 1
        if name.startswith("put"):
            post["invite"] = 1
        transitions[name] = (pre, post)
    return PetriNet(places, transitions), base_marking + (invites,)

In [8]:
N_SMALL, N_FULL = 3, 5

STATS: dict[tuple[str, int], tuple[int, int, int]] = {}

for label, builder, fname in [
    ("задача 1 — виделки одночасно", build_task1, "03-philosophers-task1.png"),
    ("задача 2 — виделки по черзі", build_task2, "04-philosophers-task2.png"),
    ("задача 3 — із запрошеннями", build_task3, "05-philosophers-task3.png"),
]:
    net_small, m_small = builder(N_SMALL)
    render(net_small.to_dot(m_small, name="phil"), ASSETS / fname)

    for n in (N_SMALL, N_FULL):
        net_i, m_i = builder(n)
        nodes_i, edges_i = net_i.reachability_graph(m_i)
        dead_i = net_i.deadlocks(nodes_i)
        STATS[(label, n)] = (len(nodes_i), len(edges_i), len(dead_i))
        record("Синтез МП", f"{label}, n={n}",
               f"розміток {len(nodes_i)}, дуг {len(edges_i)}, тупиків {len(dead_i)}")

assert STATS[("задача 1 — виделки одночасно", 3)] == (4, 6, 0)
assert STATS[("задача 1 — виделки одночасно", 5)] == (11, 30, 0)
assert STATS[("задача 2 — виделки по черзі", 3)] == (14, 27, 1)
assert STATS[("задача 2 — виделки по черзі", 5)] == (82, 265, 1)
assert STATS[("задача 3 — із запрошеннями", 3)] == (13, 24, 0)
assert STATS[("задача 3 — із запрошеннями", 5)] == (81, 260, 0)

Синтез МП | задача 1 — виделки одночасно, n=3 = розміток 4, дуг 6, тупиків 0  
Синтез МП | задача 1 — виделки одночасно, n=5 = розміток 11, дуг 30, тупиків 0  


Синтез МП | задача 2 — виделки по черзі, n=3 = розміток 14, дуг 27, тупиків 1  
Синтез МП | задача 2 — виделки по черзі, n=5 = розміток 82, дуг 265, тупиків 1  


Синтез МП | задача 3 — із запрошеннями, n=3 = розміток 13, дуг 24, тупиків 0  
Синтез МП | задача 3 — із запрошеннями, n=5 = розміток 81, дуг 260, тупиків 0  


### Тупик задачі 2 — не декларуємо, а показуємо

Знаходимо тупикову розмітку в графі досяжності й відновлюємо
найкоротшу послідовність спрацювань, що до неї веде (обхід у ширину по
вже побудованих дугах графа).

In [9]:
from collections import deque

net2, m2 = build_task2(N_SMALL)
nodes2, edges2 = net2.reachability_graph(m2)
(deadlock,) = net2.deadlocks(nodes2)

print("Тупикова розмітка:")
for place, count in zip(net2.places, deadlock):
    if count:
        print(f"   {place} = {count}")

parent = {m2: None}
queue = deque([m2])
while queue:
    cur = queue.popleft()
    for src, t, dst in edges2:
        if src == cur and dst not in parent:
            parent[dst] = (cur, t)
            queue.append(dst)
path, node = [], deadlock
while parent[node] is not None:
    prev, t = parent[node]
    path.append(t)
    node = prev
path.reverse()

record("Синтез МП", "послідовність до тупика (задача 2, n=3)", " -> ".join(path))
record("Синтез МП", "довжина шляху до тупика", len(path))
print("\nПослідовність спрацювань:", " -> ".join(path))

assert all(deadlock[net2.index[f"left{i}"]] == 1 for i in range(N_SMALL))
assert all(deadlock[net2.index[f"fork{i}"]] == 0 for i in range(N_SMALL))
assert all(deadlock[net2.index[f"eat{i}"]] == 0 for i in range(N_SMALL))

Тупикова розмітка:
   left0 = 1
   left1 = 1
   left2 = 1
Синтез МП | послідовність до тупика (задача 2, n=3) = takeL0 -> takeL1 -> takeL2  
Синтез МП | довжина шляху до тупика = 3  

Послідовність спрацювань: takeL0 -> takeL1 -> takeL2


Послідовність `takeL0 -> takeL1 -> takeL2` буквально й показує тупик:
кожен філософ по черзі бере свою ліву виделку, і після третього кроку всі
три виделки розібрані як ліві — жодному філософу вже нема чим узяти праву.
Жоден із переходів `takeR_i` та `put_i` більше не дозволений — це і є
тупикова розмітка.

### Що саме змінили запрошення

Граф досяжності для 5 філософів завеликий для рисунка (82 і 81 вершина
відповідно для задач 2 і 3 при n=5), тож у звіт іде **зменшена модель на
3 філософів** — вона має ту саму структуру, лише менша й читабельна.
Числа для $n = 5$ подано в таблиці вище (з `record()`), а не рисунком.

In [10]:
net3, m3 = build_task3(N_SMALL)
nodes3, edges3 = net3.reachability_graph(m3)

render(PetriNet.graph_to_dot(nodes2, edges2, name="reach_task2", rankdir="TB"),
             ASSETS / "06-reachability-task2-3phil.png")
render(PetriNet.graph_to_dot(nodes3, edges3, name="reach_task3", rankdir="TB"),
             ASSETS / "07-reachability-task3-3phil.png")

record("Синтез МП", "розміток: задача 2 -> задача 3 (n=3)", f"{len(nodes2)} -> {len(nodes3)}",
       "прибрано рівно одну — тупикову")
record("Синтез МП", "тупиків: задача 2 -> задача 3 (n=3)",
       f"{len(net2.deadlocks(nodes2))} -> {len(net3.deadlocks(nodes3))}")

nodes2_5, _, dead2_5 = STATS[("задача 2 — виделки по черзі", 5)]
nodes3_5, _, dead3_5 = STATS[("задача 3 — із запрошеннями", 5)]
record("Синтез МП", "розміток: задача 2 -> задача 3 (n=5)", f"{nodes2_5} -> {nodes3_5}",
       "прибрано рівно одну — тупикову")
record("Синтез МП", "тупиків: задача 2 -> задача 3 (n=5)", f"{dead2_5} -> {dead3_5}")

assert len(nodes2) - len(nodes3) == 1
assert net3.deadlocks(nodes3) == []
assert nodes2_5 - nodes3_5 == 1
assert dead3_5 == 0

print("Найпереконливіший результат синтезу:")
print(f"   n=3: {len(nodes2)} -> {len(nodes3)} розміток, тупиків {len(net2.deadlocks(nodes2))} -> {len(net3.deadlocks(nodes3))}")
print(f"   n=5: {nodes2_5} -> {nodes3_5} розміток, тупиків {dead2_5} -> {dead3_5}")

Синтез МП | розміток: задача 2 -> задача 3 (n=3) = 14 -> 13  прибрано рівно одну — тупикову
Синтез МП | тупиків: задача 2 -> задача 3 (n=3) = 1 -> 0  
Синтез МП | розміток: задача 2 -> задача 3 (n=5) = 82 -> 81  прибрано рівно одну — тупикову
Синтез МП | тупиків: задача 2 -> задача 3 (n=5) = 1 -> 0  
Найпереконливіший результат синтезу:
   n=3: 14 -> 13 розміток, тупиків 1 -> 0
   n=5: 82 -> 81 розміток, тупиків 1 -> 0


`14 → 13` і `82 → 81` — семафор запрошень прибирає з графа досяжності
рівно **одну** розмітку, і це саме та розмітка, у якій усі філософи
тримають ліву виделку й чекають на праву. Жодна інша поведінка не
втрачена: усе, що було досяжним у задачі 2 (крім тупика), лишається
досяжним і в задачі 3, а `n - 1` дозволів гарантує, що бодай один
філософ завжди може з'їсти обома виделками одразу.

Контрольна перевірка: якщо не обмежувати запрошення взагалі
(`invites=n`), тупик задачі 2 повертається — бо тоді всі $n$ філософів
знову можуть одночасно взяти ліву виделку.

In [11]:
net3_full, m3_full = build_task3(N_SMALL, invites=N_SMALL)
nodes3_full, _ = net3_full.reachability_graph(m3_full)
dead3_full = net3_full.deadlocks(nodes3_full)

record("Синтез МП", "задача 3 з invites=n (контроль)", f"тупиків {len(dead3_full)}",
       "без обмеження запрошень тупик задачі 2 повертається")
assert len(dead3_full) == 1

Синтез МП | задача 3 з invites=n (контроль) = тупиків 1  без обмеження запрошень тупик задачі 2 повертається


## Зведення результатів

Усі числа звіту беруться з цього файла, а не переписуються руками.

In [12]:
import csv

with open(DATA / "results-summary.csv", "w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=["розділ", "показник", "значення", "примітка"])
    writer.writeheader()
    writer.writerows(RESULTS)
print(f"записано {len(RESULTS)} рядків у {DATA / 'results-summary.csv'}")

записано 20 рядків у data/results-summary.csv
